In [ ]:
!pip install fastparquet

In [2]:
import numpy as np
from sklearn.metrics import confusion_matrix
import evaluate

# ==== METRICS ====
metric_acc = evaluate.load("accuracy")
metric_prec = evaluate.load("precision")
metric_rec = evaluate.load("recall")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Accuracy
    accuracy = metric_acc.compute(predictions=preds, references=labels)["accuracy"]

    # Plain precision, recall, F1 (no averaging argument)
    precision = metric_prec.compute(predictions=preds, references=labels)["precision"]
    recall = metric_rec.compute(predictions=preds, references=labels)["recall"]
    f1 = metric_f1.compute(predictions=preds, references=labels)["f1"]

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    cm_flat = cm.flatten().tolist()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm_flat
    }

# Wrapper to evaluate your sklearn logistic regression using your compute_metrics function
def evaluate_logreg_with_compute_metrics(clf, X, y, compute_metrics_fn):
    # sklearn predicts probabilities
    probs = clf.predict_proba(X)  # shape [N, 2] for binary
    # "Logits" here can just be the probabilities
    eval_pred = (probs, y)
    return compute_metrics_fn(eval_pred)


In [4]:
#classifier that determines wheter a Q is answerable or unanswerable

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModel

# mean pooling from https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# The dataset loader
def load_data():
    splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
    df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"], engine='fastparquet')
    df_val   = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"], engine='fastparquet')

    # Train sets on languages
    df_train_ar = df_train[df_train['lang'] == 'ar']
    df_train_ko = df_train[df_train['lang'] == 'ko']
    df_train_te = df_train[df_train['lang'] == 'te']

    # Validation sets on languages
    df_val_ar = df_val[df_val['lang'] == 'ar']
    df_val_ko = df_val[df_val['lang'] == 'ko']
    df_val_te = df_val[df_val['lang'] == 'te']

    return {
        "ar": (df_train_ar, df_val_ar),
        "ko": (df_train_ko, df_val_ko),
        "te": (df_train_te, df_val_te),
    }

# label extraction: 1=answerable, 0=unanswerable ---
def get_label_from_row(row):
    # True => 1 (answerable), False => 0 (unanswerable), None => no label
    v = row["answerable"]
    return 1 if v is True else 0 if v is False else None

# encoder class using sentence-transformers
class STEncoder:
    def __init__(self, name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                 device='cpu', max_len=256):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.net = AutoModel.from_pretrained(name).to(device)
        self.net.eval()
        self.device = device
        self.max_len = max_len

    @torch.no_grad()
    def encode(self, texts, batch_size=64, normalize=True):
        out = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = self.tok(batch, padding=True, truncation=True,
                           max_length=self.max_len, return_tensors='pt').to(self.device)
            model_output = self.net(**enc)
            sent_emb = mean_pooling(model_output, enc['attention_mask'])  # YOUR function
            if normalize:
                sent_emb = torch.nn.functional.normalize(sent_emb, p=2, dim=1)
            out.append(sent_emb.cpu().numpy())
        return np.vstack(out)


# Build (X, y) with 3 features, cosine similarity, L2 and L1 distances
def build_xy(df, encoder, max_n=None):
    import numpy as np

    df = df[df["question"].notna() & df["context"].notna()].copy()
    if max_n is not None:
        df = df.head(max_n)

    y = df.apply(get_label_from_row, axis=1)
    df = df[y.notna()]
    y = y[y.notna()].astype(int).values
    if len(df) == 0:
        return None, None

    qs = df["question"].astype(str).tolist()
    cs = df["context"].astype(str).tolist()

    # --- embeddings ---
    q_emb = encoder.encode(qs)  # [N, D]
    c_emb = encoder.encode(cs)  # [N, D]

    # --- cosine (robust even if normalize=False later) ---
    eps  = 1e-9
    dots = np.einsum("nd,nd->n", q_emb, c_emb)[:, None]           # [N,1]
    qn   = np.linalg.norm(q_emb, axis=1, keepdims=True)           # [N,1]
    cn   = np.linalg.norm(c_emb, axis=1, keepdims=True)           # [N,1]
    cos  = np.clip(dots / np.clip(qn * cn, eps, None), -1.0, 1.0) # [N,1]

    # --- distances ---
    diff = q_emb - c_emb
    l2   = np.linalg.norm(diff, axis=1, keepdims=True)            # [N,1]
    l1   = np.linalg.norm(diff, ord=1, axis=1, keepdims=True)     # [N,1]

    # final feature matrix: [cos, l2, l1]
    X = np.hstack([cos, l2, l1])                                   # [N,3]
    return X, y

def main():
    data = load_data()
    enc = STEncoder(device='cpu', max_len=256)  # simple CPU setup

    results = []
    for lang in ["ar", "ko", "te"]:
        dtr, dva = data[lang]
        if dtr.empty or dva.empty:
            print(f"[{lang}] skipped (empty split)")
            continue

        print(f"\n=== {lang} ===")
        # set max_n to a smaller number while testing if CPU is slow (e.g., 3000)
        Xtr, ytr = build_xy(dtr, enc, max_n=None)
        Xva, yva = build_xy(dva, enc, max_n=None)
        if Xtr is None or Xva is None:
            print(f"[{lang}] no labeled pairs found")
            continue

        clf = LogisticRegression(max_iter=200, class_weight="balanced")
        clf.fit(Xtr, ytr)
        p = clf.predict(Xva)

        # Overall metrics
        acc = accuracy_score(yva, p)
        f1  = f1_score(yva, p)

        # Per-class accuracies (subset accuracy on each class)
        # answerable = 1, unanswerable = 0
        mask_pos   = (yva == 1)
        mask_neg   = (yva == 0)
        ans_acc    = (p[mask_pos] == 1).mean() if mask_pos.sum() > 0 else float("nan")
        unans_acc  = (p[mask_neg] == 0).mean() if mask_neg.sum() > 0 else float("nan")

        print(
            f"[{lang}] val acc={acc:.3f}  F1={f1:.3f}  "
            f"AnsAcc={ans_acc:.3f}  UnansAcc={unans_acc:.3f}  "
            f"n={len(yva)}"
        )
        
        # --- Evaluate using your compute_metrics function ---
        metrics_results = evaluate_logreg_with_compute_metrics(clf, Xva, yva, compute_metrics)
        print(f"[{lang}] compute_metrics results:")
        for k, v in metrics_results.items():
            print(f"    {k}: {v}")

        # keep both per-class metrics for the summary
        results.append((lang, acc, f1, ans_acc, unans_acc, len(yva)))

    print("\nSummary:")
    for lang, acc, f1, ans_acc, unans_acc, n in results:
        print(
            f"{lang}: acc={acc:.3f}  F1={f1:.3f}  "
            f"AnsAcc={ans_acc:.3f}  UnansAcc={unans_acc:.3f}  n={n}"
        )
if __name__ == "__main__":
    main()




=== ar ===


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath

[ar] val acc=0.470  F1=0.600  AnsAcc=0.455  UnansAcc=0.577  n=415
[ar] compute_metrics results:
    accuracy: 0.46987951807228917
    precision: 0.8823529411764706
    recall: 0.45454545454545453
    f1: 0.6
    confusion_matrix: [30, 22, 198, 165]

=== ko ===


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath

[ko] val acc=0.419  F1=0.568  AnsAcc=0.404  UnansAcc=0.684  n=356
[ko] compute_metrics results:
    accuracy: 0.41853932584269665
    precision: 0.9577464788732394
    recall: 0.4035608308605341
    f1: 0.5678496868475992
    confusion_matrix: [13, 6, 201, 136]

=== te ===


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/utils/extmath

[te] val acc=0.547  F1=0.634  AnsAcc=0.519  UnansAcc=0.634  n=384
[te] compute_metrics results:
    accuracy: 0.546875
    precision: 0.8162162162162162
    recall: 0.5189003436426117
    f1: 0.634453781512605
    confusion_matrix: [59, 34, 140, 151]

Summary:
ar: acc=0.470  F1=0.600  AnsAcc=0.455  UnansAcc=0.577  n=415
ko: acc=0.419  F1=0.568  AnsAcc=0.404  UnansAcc=0.684  n=356
te: acc=0.547  F1=0.634  AnsAcc=0.519  UnansAcc=0.634  n=384
